# Phase 4 — Receiver-normalised source (thesis 3.2)

Every WPD feature vector is **centred** per feature on the mean of the noise-only windows of its own receiver
session (`rfml_uav.data.normalise`), **at training and at test time**. Dividing by the noise std as well was the
first variant; it is kept as an ablation (`phase4_std/`, `checkpoints/paper3/noisenorm_std/`), see design D1/D7. Two estimators: *session* (all noise windows of
the recording) and *stream* (EMA over noise windows in time order, β = 0.01, warm-up k = 50).
Phase 1 protocol otherwise unchanged (same folds), compared side by side with the Phase 1 source on E1 / E2 / E3.

In [1]:
import os
import json
import itertools
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from scipy.special import softmax
from sklearn.metrics import f1_score
from tqdm import tqdm

from rfml_uav.data.impairments import SEVERITY
from rfml_uav.data.normalise import session_normalise, stream_normalise
from rfml_uav.drone_detect.dataset import DRONES
from rfml_uav.eval.metrics import classification_report_dict
from rfml_uav.eval.splits import make_splits, undersample
from rfml_uav.eval.stats import paired_test
from rfml_uav.nn.models import get_model
from rfml_uav.nn.train import train_best_val, predict_logits

load_dotenv()
DATA = os.getenv("DATA_PATH")
P1, P2, P3 = (os.path.join(DATA, f"results/paper3/phase{i}") for i in (1, 2, 3))
P4_STD = os.path.join(DATA, "results/paper3/phase4_std")  # rejected std-normalised variant (ablation)
out, ckpt_path = os.path.join(DATA, "results/paper3/phase4"), os.path.join(DATA, "checkpoints/paper3/noisenorm")
for p in (os.path.join(out, "runs"), ckpt_path):
    os.makedirs(p, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSES = list(DRONES)
GROUP = {"AIR": "A", "DIS": "A", "PHA": "A", "INS": "B", "MIN": "B", "MP1": "B", "MP2": "B"}  # receiver configuration
SUBSETS, MODES = ["CLEAN", "BLUE", "WIFI", "BOTH"], ["ON", "HO", "FY"]
SEEDS, K_FOLDS = [0, 1, 2, 3, 4], 5
EPOCHS, BATCH_SIZE, LEARNING_RATE = 50, 256, 0.01   # Phase 1
BETA, K_WARM = 0.01, 50
CELLS = [(k, s) for k in SEVERITY for s in (1, 2, 3)] + [("tilt", 4)]

In [2]:
F = dict(np.load(os.path.join(P1, "features.npz")))
F["y"] = np.array([CLASSES.index(d) for d in F["drone"]])
clean, active = F["subset"] == "CLEAN", F["active"]
seen = {(d, m) for d, m in zip(F["drone"][clean], F["mode"][clean])}
F["seen"] = np.array([(d, m) in seen for d, m in zip(F["drone"], F["mode"])])
REC = pd.Series(np.arange(len(F["y"]))).groupby(F["rec_id"]).apply(np.array).to_dict()  # rec -> rows, time-ordered
assert all(np.all(np.diff(F["start"][r][F["active"][r]]) > 0) for r in REC.values())


def stream_all(X, rows_by_rec, is_noise, beta=BETA):
    """Stream-normalise each recording separately (cold start per recording); rows in time order."""
    Xn, valid = np.zeros((len(X), X.shape[1]), np.float32), np.zeros(len(X), bool)
    for rows in rows_by_rec:
        Xn[rows], valid[rows] = stream_normalise(X[rows], is_noise[rows], beta=beta, k=K_WARM)
    return Xn, valid


XS = session_normalise(F["X"], F["rec_id"], ~active)
order = [r[np.argsort(F["start"][r], kind="stable")] for r in REC.values()]  # active and noise rows interleaved by time
XT, VALID = stream_all(F["X"], order, ~active)
ACTIVE_ROWS = np.flatnonzero(active)
print("stream warm-up excludes {:.1%} of active rows".format(1 - VALID[ACTIVE_ROWS].mean()))
XS_t = torch.tensor(XS, device=device).unsqueeze(1)
y_t = torch.tensor(F["y"], device=device)

stream warm-up excludes 6.3% of active rows


## Same folds as Phase 1

In [3]:
def source_rows(data, seed):
    return undersample(np.flatnonzero(clean & (active if data == "active" else ~active)), F["y"], seed)


for seed in SEEDS:
    idx = source_rows("active", seed)
    for fold, (tr, va, te) in enumerate(make_splits(F["y"][idx], F["rec_id"][idx], "recording", n_folds=K_FOLDS, seed=seed)):
        assert np.array_equal(idx[te], np.load(os.path.join(P1, f"runs/active_linear_s{seed}_f{fold}.npz"))["test_idx"])
print("folds identical to Phase 1 for all 25 (seed, fold)")

folds identical to Phase 1 for all 25 (seed, fold)


## Train: active (linear tail, the source) and noise-only (default tail, the shortcut re-test, as in Phase 1)

In [4]:
TARGET = np.flatnonzero(active & ~clean)
TAIL = {"active": "linear", "noise": "default"}


def run_path(data, seed, fold):
    return os.path.join(out, "runs", f"{data}_{TAIL[data]}_s{seed}_f{fold}.npz")


def logits(model, X):
    return predict_logits(model.eval(), torch.as_tensor(X, device=device).unsqueeze(1) if not torch.is_tensor(X) else X).cpu().numpy()


for data, seed in tqdm(list(itertools.product(["active", "noise"], SEEDS))):
    idx = source_rows(data, seed)
    for fold, (tr, va, te) in enumerate(make_splits(F["y"][idx], F["rec_id"][idx], "recording", n_folds=K_FOLDS, seed=seed)):
        if os.path.exists(run_path(data, seed, fold)):
            continue
        torch.manual_seed(seed)
        model = get_model("CNN", input_dim=256, n_classes=len(CLASSES), linear_tail=TAIL[data] == "linear").to(device)
        model, val_f1, best_epoch = train_best_val(model, XS_t, y_t, idx[tr], idx[va], epochs=EPOCHS, lr=LEARNING_RATE,
                                                   batch_size=BATCH_SIZE, seed=seed)
        res = {"test_idx": idx[te], "test_logits": logits(model, XS_t[idx[te]]), "val_f1": val_f1, "best_epoch": best_epoch}
        if data == "active":
            torch.save(model.state_dict(), os.path.join(ckpt_path, f"active_linear_s{seed}_f{fold}.pt"))
            res.update(tgt_idx=TARGET, tgt_logits=logits(model, XS_t[TARGET]), stream_rows=ACTIVE_ROWS,
                       stream_logits=logits(model, XT[ACTIVE_ROWS]), stream_valid=VALID[ACTIVE_ROWS])
        np.savez_compressed(run_path(data, seed, fold), **res)
assert len([f for f in os.listdir(ckpt_path) if f.endswith(".pt")]) == 25

  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:00<00:01,  7.54it/s]

 20%|██        | 2/10 [00:00<00:01,  7.51it/s]

 30%|███       | 3/10 [00:00<00:00,  7.51it/s]

 40%|████      | 4/10 [00:00<00:00,  7.54it/s]

...

 70%|███████   | 7/10 [00:00<00:00,  7.57it/s]

 80%|████████  | 8/10 [00:01<00:00,  7.58it/s]

 90%|█████████ | 9/10 [00:01<00:00,  7.59it/s]

100%|██████████| 10/10 [00:01<00:00,  7.58it/s]

100%|██████████| 10/10 [00:01<00:00,  7.56it/s]

## Shortcut re-test (noise-only windows, session-normalised)

In [5]:
g = np.array([GROUP[c] for c in CLASSES])
sc = []
for seed, fold in itertools.product(SEEDS, range(K_FOLDS)):
    r = np.load(run_path("noise", seed, fold))
    y, pred = F["y"][r["test_idx"]], r["test_logits"].argmax(1)
    sc.append({"accuracy": (y == pred).mean(), "group_accuracy": (g[y] == g[pred]).mean()})
sc = pd.DataFrame(sc).agg(["mean", "std"]).T
sc["chance"] = [1 / 7, 1 / 2]
sc["phase1_mean"] = pd.read_csv(os.path.join(P1, "shortcut_test.csv"), index_col=0).loc[["accuracy", "group_accuracy"], "mean"].values
sc.to_csv(os.path.join(out, "shortcut_test.csv"))
print("noise-only 7-class accuracy {:.3f} vs 2 x chance {:.3f} -> shortcut {}".format(
    sc.loc["accuracy", "mean"], 2 / 7, "STILL flagged" if sc.loc["accuracy", "mean"] > 2 / 7 else "removed"))
sc.round(4)

noise-only 7-class accuracy 0.317 vs 2 x chance 0.286 -> shortcut STILL flagged


,mean,std,chance,phase1_mean
accuracy,0.3169,0.046,0.1429,0.4311
group_accuracy,0.9074,0.033,0.5000,0.9772


## E1: interference, side by side

In [6]:
def pm(df, cols, by):
    gr = df.groupby(by, sort=False)[cols].agg(["mean", "std"])
    return pd.DataFrame({c: gr[(c, "mean")].map("{:.4f}".format) + " ± " + gr[(c, "std")].map("{:.4f}".format) for c in cols})


def scores(rows, z):
    y, p = F["y"][rows], z.argmax(1)
    return {"accuracy": (y == p).mean(), "macro_f1": f1_score(y, p, labels=np.unique(y), average="macro")}


e1, e1_mode = [], []
for seed, fold in itertools.product(SEEDS, range(K_FOLDS)):
    p1 = np.load(os.path.join(P1, f"runs/active_linear_s{seed}_f{fold}.npz"))
    p4 = np.load(run_path("active", seed, fold))
    idx = {"phase1": np.r_[p1["test_idx"], p1["tgt_idx"]], "session": np.r_[p4["test_idx"], p4["tgt_idx"]]}
    z = {"phase1": np.r_[p1["test_logits"], p1["tgt_logits"]], "session": np.r_[p4["test_logits"], p4["tgt_logits"]]}
    assert np.array_equal(idx["phase1"], idx["session"])
    rows_all = idx["phase1"]
    pos = np.searchsorted(p4["stream_rows"], rows_all)
    z["stream"], ok = p4["stream_logits"][pos], p4["stream_valid"][pos]
    for sub in SUBSETS:
        m = (F["subset"][rows_all] == sub) & F["seen"][rows_all]
        if sub == "CLEAN":
            m = np.zeros(len(rows_all), bool); m[:len(p1["test_idx"])] = True
        for name, mask, src in [("phase1", m, "phase1"), ("session", m, "session"), ("stream", m & ok, "stream"),
                                ("phase1_on_stream_rows", m & ok, "phase1")]:
            e1.append({"seed": seed, "fold": fold, "subset": sub, "source": name, "excluded": 1 - mask.sum() / m.sum(),
                       **scores(rows_all[mask], z[src][mask])})
            for mode in MODES:
                mm = mask & (F["mode"][rows_all] == mode)
                if mm.any():
                    e1_mode.append({"seed": seed, "fold": fold, "subset": sub, "mode": mode, "source": name,
                                    **scores(rows_all[mm], z[src][mm])})
e1, e1_mode = pd.DataFrame(e1), pd.DataFrame(e1_mode)


def side_by_side(df, keys, path):
    """mean ± std per source + normalised − Phase 1 (paired by run; stream vs Phase 1 on the same valid rows)."""
    piv = df.pivot_table(index=keys + ["seed", "fold"], columns="source", values="macro_f1")
    rows = []
    for key, gr in piv.groupby(level=list(range(len(keys)))):
        key = key if isinstance(key, tuple) else (key,)
        row = dict(zip(keys, key))
        for src in ("phase1", "session", "stream"):
            row[src] = f"{gr[src].mean():.4f} ± {gr[src].std():.4f}"
        for src, ref in (("session", "phase1"), ("stream", "phase1_on_stream_rows")):
            t = paired_test(gr[src], gr[ref])
            row.update({f"{src}_diff": t["diff"], f"{src}_ci_low": t["ci_lo"], f"{src}_ci_high": t["ci_hi"], f"{src}_p": t["p"]})
        rows.append(row)
    res = pd.DataFrame(rows).set_index(keys)
    if "excluded" in df:
        res["stream_excluded"] = df[df.source == "stream"].groupby(keys)["excluded"].mean()
    res.to_csv(path)
    return res


def with_ablation(table, name, keys):
    """Append the std-normalised variant's session column and its difference vs Phase 1 (saved Phase 4 std run)."""
    std = pd.read_csv(os.path.join(P4_STD, name), index_col=list(range(len(keys))))
    table = table.join(std[["session", "session_diff", "session_ci_low", "session_ci_high", "session_p"]]
                       .rename(columns=lambda c: "std_" + c))
    table.to_csv(os.path.join(out, name))
    return table


E1 = with_ablation(side_by_side(e1, ["subset"], os.path.join(out, "e1_table.csv")), "e1_table.csv", ["subset"]).reindex(SUBSETS)
side_by_side(e1_mode, ["subset", "mode"], os.path.join(out, "e1_by_mode.csv"))
assert len(E1) == 4 and E1[["session_diff", "stream_diff", "std_session_diff"]].notna().all().all()
E1.round(4)

/tmp/ipykernel_55799/3017278413.py:41: FutureWarning: Creating a Groupby object with a length-1 list-like level parameter will yield indexes as tuples in a future version. To keep indexes as scalars, create Groupby objects with a scalar level parameter instead.
  for key, gr in piv.groupby(level=list(range(len(keys)))):


,phase1,session,stream,session_diff,session_ci_low,session_ci_high,session_p,stream_diff,stream_ci_low,stream_ci_high,stream_p,stream_excluded,std_session,std_session_diff,std_session_ci_low,std_session_ci_high,std_session_p
subset,,,,,,,,,,,,,,,,,
CLEAN,0.7274 ± 0.0479,0.8214 ± 0.0411,0.8180 ± 0.0354,0.0940,0.0772,0.1106,0.0,0.0877,0.0702,0.1046,0.0,0.0763,0.7020 ± 0.0673,-0.0254,-0.0500,-0.0027,0.0802
BLUE,0.5829 ± 0.0105,0.6952 ± 0.0164,0.6849 ± 0.0171,0.1124,0.1056,0.1188,0.0,0.1015,0.0945,0.1083,0.0,0.0588,0.6264 ± 0.0238,0.0436,0.0352,0.0520,0.0000
WIFI,0.5876 ± 0.0122,0.7049 ± 0.0203,0.6937 ± 0.0178,0.1173,0.1079,0.1261,0.0,0.1108,0.1025,0.1187,0.0,0.0603,0.5870 ± 0.0329,-0.0005,-0.0129,0.0114,0.9789
BOTH,0.4618 ± 0.0076,0.5873 ± 0.0206,0.5797 ± 0.0201,0.1255,0.1164,0.1343,0.0,0.1185,0.1097,0.1271,0.0,0.0544,0.4914 ± 0.0209,0.0296,0.0219,0.0375,0.0000


## E2: receiver impairments, side by side (impaired active rows normalised by impaired noise rows)

In [7]:
MODELS = {}
for seed, fold in itertools.product(SEEDS, range(K_FOLDS)):
    m = get_model("CNN", input_dim=256, n_classes=len(CLASSES), linear_tail=True)
    m.load_state_dict(torch.load(os.path.join(ckpt_path, f"active_linear_s{seed}_f{fold}.pt"), map_location=device))
    MODELS[(seed, fold)] = m.to(device).eval()


def normalised_cell(cell, beta=BETA):
    """Impaired CLEAN rows (active + noise) of a cell -> (rows, session X, stream X, stream valid, is_active)."""
    a = np.load(os.path.join(P2, "dronedetect", f"{cell[0]}_{cell[1]}.npz"))
    n = np.load(os.path.join(P2, "dronedetect_noise", f"{cell[0]}_{cell[1]}.npz"))
    rows, X = np.r_[a["row"], n["row"]], np.r_[a["X"], n["X"]]
    o = np.argsort(rows)
    rows, X = rows[o], X[o]
    is_noise = ~F["active"][rows]
    Xs = session_normalise(X, F["rec_id"][rows], is_noise)
    by_rec = pd.Series(np.arange(len(rows))).groupby(F["rec_id"][rows]).apply(np.array)
    Xt, ok = stream_all(X, [r[np.argsort(F["start"][rows][r], kind="stable")] for r in by_rec], is_noise, beta)
    return rows, Xs, Xt, ok


e2 = []
for cell in tqdm(CELLS):
    rows, Xs, Xt, ok = normalised_cell(cell)
    for (seed, fold), model in MODELS.items():
        te = np.load(run_path("active", seed, fold))["test_idx"]
        pos = np.searchsorted(rows, te)
        assert np.array_equal(rows[pos], te)
        ref = np.load(os.path.join(P2, f"runs/dronedetect_s{seed}_f{fold}.npz"))
        assert np.array_equal(ref["test_idx"], te)
        z1, zs, zt = ref[f"{cell[0]}_{cell[1]}"], logits(model, Xs[pos]), logits(model, Xt[pos])
        v = ok[pos]
        for name, mask, z in [("phase1", np.ones(len(te), bool), z1), ("session", np.ones(len(te), bool), zs),
                              ("stream", v, zt), ("phase1_on_stream_rows", v, z1)]:
            e2.append({"seed": seed, "fold": fold, "cell": f"{cell[0]}_{cell[1]}", "source": name,
                       "excluded": 1 - mask.mean(), **scores(te[mask], z[mask])})
e2 = pd.DataFrame(e2)
E2 = with_ablation(side_by_side(e2, ["cell"], os.path.join(out, "e2_table.csv")), "e2_table.csv", ["cell"])
E2 = E2.reindex([f"{k}_{s}" for k, s in CELLS])
assert len(E2) == 13 and E2["std_session_diff"].notna().all()
E2.round(4)

  0%|          | 0/13 [00:00<?, ?it/s]

  8%|▊         | 1/13 [00:01<00:20,  1.74s/it]

 15%|█▌        | 2/13 [00:03<00:18,  1.67s/it]

 23%|██▎       | 3/13 [00:04<00:16,  1.65s/it]

 31%|███       | 4/13 [00:06<00:14,  1.64s/it]

...

 92%|█████████▏| 12/13 [00:19<00:01,  1.62s/it]

100%|██████████| 13/13 [00:21<00:00,  1.63s/it]

100%|██████████| 13/13 [00:21<00:00,  1.63s/it]


/tmp/ipykernel_55799/3017278413.py:41: FutureWarning: Creating a Groupby object with a length-1 list-like level parameter will yield indexes as tuples in a future version. To keep indexes as scalars, create Groupby objects with a scalar level parameter instead.
  for key, gr in piv.groupby(level=list(range(len(keys)))):


,phase1,session,stream,session_diff,session_ci_low,session_ci_high,session_p,stream_diff,stream_ci_low,stream_ci_high,stream_p,stream_excluded,std_session,std_session_diff,std_session_ci_low,std_session_ci_high,std_session_p
cell,,,,,,,,,,,,,,,,,
awgn_1,0.6653 ± 0.0454,0.8111 ± 0.0421,0.8108 ± 0.0392,0.1458,0.1260,0.1649,0.0000,0.1432,0.1232,0.1625,0.0000,0.0763,0.6944 ± 0.0669,0.0291,0.0083,0.0495,0.0203
awgn_2,0.5424 ± 0.0559,0.7088 ± 0.0429,0.7110 ± 0.0434,0.1665,0.1433,0.1887,0.0000,0.1663,0.1430,0.1888,0.0000,0.0763,0.6143 ± 0.0713,0.0719,0.0450,0.1012,0.0000
awgn_3,0.4092 ± 0.0601,0.5610 ± 0.0423,0.5615 ± 0.0447,0.1518,0.1256,0.1770,0.0000,0.1507,0.1248,0.1756,0.0000,0.0763,0.4264 ± 0.0960,0.0172,-0.0200,0.0575,0.5077
gain_1,0.7201 ± 0.0473,0.1701 ± 0.0720,0.1697 ± 0.0719,-0.5500,-0.5779,-0.5222,0.0000,-0.5530,-0.5805,-0.5250,0.0000,0.0763,0.4937 ± 0.1067,-0.2264,-0.2718,-0.1809,0.0000
gain_2,0.7057 ± 0.0489,0.1190 ± 0.0454,0.1175 ± 0.0439,-0.5867,-0.6049,-0.5674,0.0000,-0.5896,-0.6076,-0.5704,0.0000,0.0763,0.4423 ± 0.0991,-0.2634,-0.3098,-0.2175,0.0000
gain_3,0.6720 ± 0.0544,0.1124 ± 0.0476,0.1109 ± 0.0459,-0.5596,-0.5822,-0.5352,0.0000,-0.5626,-0.5856,-0.5382,0.0000,0.0763,0.4129 ± 0.0841,-0.2591,-0.3003,-0.2174,0.0000
tilt_1,0.7246 ± 0.0466,0.8166 ± 0.0426,0.8115 ± 0.0396,0.0921,0.0752,0.1080,0.0000,0.0844,0.0659,0.1011,0.0000,0.0763,0.6975 ± 0.0668,-0.0271,-0.0517,-0.0047,0.0516
tilt_2,0.7186 ± 0.0441,0.7743 ± 0.0634,0.7706 ± 0.0618,0.0557,0.0307,0.0789,0.0006,0.0495,0.0227,0.0738,0.0028,0.0763,0.7029 ± 0.0701,-0.0158,-0.0388,0.0053,0.3123
tilt_3,0.6994 ± 0.0410,0.5738 ± 0.0653,0.5698 ± 0.0618,-0.1256,-0.1511,-0.0989,0.0000,-0.1330,-0.1592,-0.1070,0.0000,0.0763,0.6886 ± 0.0650,-0.0108,-0.0323,0.0101,0.3810


## Realistic AGC: rare gain steps (design D8)

The E2 gain cells put a step in **every** window, noise windows included. Here only a fixed 5% of CLEAN rows
(active and noise alike, the same rows for both sources) take their `gain_s` features; the rest stay clean.

In [8]:
P1_MODELS = {}
for seed, fold in itertools.product(SEEDS, range(K_FOLDS)):
    m = get_model("CNN", input_dim=256, n_classes=len(CLASSES), linear_tail=True)
    m.load_state_dict(torch.load(os.path.join(DATA, f"checkpoints/paper3/phase1/active_linear_s{seed}_f{fold}.pt"), map_location=device))
    P1_MODELS[(seed, fold)] = m.to(device).eval()

CLEAN_ROWS = np.flatnonzero(clean)
STEPPED = np.array([np.random.default_rng([0, 99, r]).random() < 0.05 for r in CLEAN_ROWS])
print(f"{STEPPED.mean():.2%} of CLEAN rows stepped")
agc = []
for s in (1, 2, 3):
    a = np.load(os.path.join(P2, "dronedetect", f"gain_{s}.npz")); n = np.load(os.path.join(P2, "dronedetect_noise", f"gain_{s}.npz"))
    imp_rows, imp_X = np.r_[a["row"], n["row"]], np.r_[a["X"], n["X"]]
    X = F["X"][CLEAN_ROWS].copy()
    pos = np.searchsorted(CLEAN_ROWS, imp_rows)
    X[pos[STEPPED[pos]]] = imp_X[STEPPED[pos]]
    Xc = session_normalise(X, F["rec_id"][CLEAN_ROWS], ~active[CLEAN_ROWS])
    for (seed, fold), model in MODELS.items():
        te = np.load(run_path("active", seed, fold))["test_idx"]
        q = np.searchsorted(CLEAN_ROWS, te)
        agc.append({"severity": s, "seed": seed, "fold": fold,
                    "phase1": scores(te, logits(P1_MODELS[(seed, fold)], X[q]))["macro_f1"],
                    "centred": scores(te, logits(model, Xc[q]))["macro_f1"]})
agc = pd.DataFrame(agc)
rows = []
for s, g in agc.groupby("severity"):
    t = paired_test(g["centred"], g["phase1"])
    rows.append({"severity": s, "step_db": SEVERITY["gain"][s], "phase1": f"{g.phase1.mean():.4f} ± {g.phase1.std():.4f}",
                 "centred": f"{g.centred.mean():.4f} ± {g.centred.std():.4f}", "diff": t["diff"], "ci_low": t["ci_lo"],
                 "ci_high": t["ci_hi"], "wilcoxon_p": t["p"]})
AGC = pd.DataFrame(rows).set_index("severity")
AGC.to_csv(os.path.join(out, "agc_rare.csv"))
assert len(AGC) == 3
AGC.round(4)

4.95% of CLEAN rows stepped


,step_db,phase1,centred,diff,ci_low,ci_high,wilcoxon_p
severity,,,,,,,
1,3,0.7271 ± 0.0477,0.8050 ± 0.0407,0.0778,0.0621,0.0934,0.0
2,6,0.7264 ± 0.0477,0.7975 ± 0.0424,0.0711,0.0548,0.0872,0.0
3,12,0.7247 ± 0.0479,0.7923 ± 0.0441,0.0676,0.0504,0.0843,0.0


## β sensitivity (stream estimator): E1 BOTH and E2 awgn 3

In [9]:
sens = []
both_rows = [r for r in order if F["subset"][r[0]] == "BOTH"]
for beta in (0.003, BETA, 0.03):
    Xt_b, ok_b = stream_all(F["X"], both_rows, ~active, beta)
    rows_c, _, Xt_c, ok_c = normalised_cell(("awgn", 3), beta)
    for (seed, fold), model in MODELS.items():
        m = np.flatnonzero((F["subset"] == "BOTH") & active & F["seen"] & ok_b)
        sens.append({"condition": "E1 BOTH", "beta": beta, "seed": seed, "fold": fold, **scores(m, logits(model, Xt_b[m]))})
        te = np.load(run_path("active", seed, fold))["test_idx"]
        pos = np.searchsorted(rows_c, te); v = ok_c[pos]
        sens.append({"condition": "E2 awgn_3", "beta": beta, "seed": seed, "fold": fold, **scores(te[v], logits(model, Xt_c[pos][v]))})
sens = pm(pd.DataFrame(sens), ["accuracy", "macro_f1"], ["condition", "beta"])
sens.to_csv(os.path.join(out, "beta_sensitivity.csv"))
assert len(sens) == 6
sens

,,accuracy,macro_f1
condition,beta,,
E1 BOTH,0.003,0.5775 ± 0.0198,0.5822 ± 0.0204
E2 awgn_3,0.003,0.5784 ± 0.0415,0.5616 ± 0.0446
E1 BOTH,0.010,0.5746 ± 0.0196,0.5797 ± 0.0201
E2 awgn_3,0.010,0.5780 ± 0.0411,0.5615 ± 0.0447
E1 BOTH,0.030,0.5679 ± 0.0181,0.5737 ± 0.0184
E2 awgn_3,0.030,0.5767 ± 0.0402,0.5605 ± 0.0436


## Adoption rule (amended)

Needs notebook 5 run with `SOURCE=noisenorm` (writes `phase3/noisenorm/runs`). The normalised source is adopted iff its
source-only macro-F1, averaged per run over the adoption targets, beats the Phase 1 source with paired Wilcoxon p < 0.05.
**Amended rule** (author decision 2026-10-19): adoption targets = E3 targets with `gain_3` (a step in every window,
including the noise windows the normalisation uses) replaced by `agc_3` (rare steps; specified before the original
rule was evaluated). The original rule's result is reported alongside.

In [10]:
E3_TARGETS = ["BLUE", "WIFI", "BOTH", "awgn_2", "awgn_3", "gain_3", "adc_3", "tilt_4"]
AMENDED = [t if t != "gain_3" else "agc_3" for t in E3_TARGETS]
norm_dir, p1_dir = os.path.join(P3, "noisenorm", "runs"), os.path.join(P3, "runs")
need = [os.path.join(d, f"{t}_s{s}_f{f}.npz") for d in (norm_dir, p1_dir) for t, s, f in itertools.product(set(E3_TARGETS + AMENDED), SEEDS, range(K_FOLDS))]
if not all(map(os.path.exists, need)):
    print("phase3 runs missing (incl. agc_3): run notebook 5 for both sources, then re-run this cell")
else:
    def pooled(d, targets):
        return np.array([np.mean([f1_score(F["y"][(r := np.load(os.path.join(d, f"{t}_s{s}_f{f}.npz")))["test_rows"]],
                                            r["source_logits"].argmax(1), average="macro") for t in targets])
                         for s, f in itertools.product(SEEDS, range(K_FOLDS))])

    def rule(targets):
        a, b = pooled(norm_dir, targets), pooled(p1_dir, targets)
        t = paired_test(a, b)
        return {"targets": targets, "adopted_source": "noisenorm" if t["diff"] > 0 and t["p"] < 0.05 else "phase1",
                "pooled_diff": t["diff"], "ci_low": t["ci_lo"], "ci_high": t["ci_hi"], "wilcoxon_p": t["p"],
                "pooled_noisenorm": a.mean(), "pooled_phase1": b.mean()}

    original, amended = rule(E3_TARGETS), rule(AMENDED)
    clean_row = E1.loc["CLEAN"]
    decision = {"adopted_source": amended["adopted_source"], "rule": "amended (gain_3 -> agc_3)",
                "amendment_reason": "gain_3 steps every window, including the noise windows the normalisation is estimated "
                                    "from; the rare-step AGC check was specified before the original rule was evaluated",
                "amended": amended, "original": original,
                "clean_diff": clean_row["session_diff"], "clean_ci": [clean_row["session_ci_low"], clean_row["session_ci_high"]]}
    with open(os.path.join(out, "adoption.json"), "w") as f:
        json.dump(decision, f, indent=1, default=float)
    print(json.dumps(decision, indent=1, default=float))

{
 "adopted_source": "noisenorm",
 "rule": "amended (gain_3 -> agc_3)",
 "amendment_reason": "gain_3 steps every window, including the noise windows the normalisation is estimated from; the rare-step AGC check was specified before the original rule was evaluated",
 "amended": {
  "targets": [
   "BLUE",
   "WIFI",
   "BOTH",
   "awgn_2",
   "awgn_3",
   "agc_3",
   "adc_3",
   "tilt_4"
  ],
  "adopted_source": "noisenorm",
  "pooled_diff": 0.0656020836489667,
  "ci_low": 0.05578117611217518,
  "ci_high": 0.0754433020902905,
  "wilcoxon_p": 5.960464477539063e-08,
  "pooled_noisenorm": 0.6469022992958972,
  "pooled_phase1": 0.5813002156469306
 },
 "original": {
  "targets": [
   "BLUE",
   "WIFI",
   "BOTH",
   "awgn_2",
   "awgn_3",
   "gain_3",
   "adc_3",
   "tilt_4"
  ],
  "adopted_source": "phase1",
  "pooled_diff": -0.011960995173714987,
  "ci_low": -0.023026576383231138,
  "ci_high": -0.0012424500491615238,
  "wilcoxon_p": 0.0709836483001709,
  "pooled_noisenorm": 0.56301880422214